# Advanced usage: backends, parallelism, zarr I/O, and CLI

This notebook covers production-grade features:

1. **Backend selection** — NumPy vs PyTorch (CPU / GPU).
2. **Parallel z-fitting** with `n_workers` in `fit_mosaic()`.
3. **OME-Zarr I/O** — writing and reading back a corrected volume.
4. **CLI** — running `basic correct`, `basic fit`, and `basic preview`
   programmatically via `subprocess`.

In [ ]:
%matplotlib inline

import shutil
import subprocess
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from linum_basic import BaSiC
from linum_basic.data import load_sample_image
from linum_basic.fit import apply_fit, fit_mosaic
from linum_basic.mosaic import MosaicGrid

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "figure.facecolor": "white",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

## 1. Backend selection

BaSiC supports two backends:

| Backend | When to use |
|---------|-------------|
| `"numpy"` (default) | CPU; no extra deps; works everywhere |
| `"torch"` | CUDA GPU acceleration on large stacks |

The `"auto"` backend picks Torch+CUDA if available, else falls back to NumPy.
Apple MPS is not supported — see the [GPU docs](../gpu.md).

In [ ]:
TILE = 64
rng = np.random.default_rng(0)

source = load_sample_image().astype(np.float32) / 255.0
h, w = source.shape
tiles = [source[r * TILE : (r + 1) * TILE, c * TILE : (c + 1) * TILE] for r in range(h // TILE) for c in range(w // TILE)]
brightness = rng.uniform(0.3, 0.9, (len(tiles), 1, 1)).astype(np.float32)
y, x = np.mgrid[-1 : 1 : TILE * 1j, -1 : 1 : TILE * 1j]  # type: ignore[misc]
vignette = (0.5 + 0.5 * np.exp(-1.5 * (x**2 + y**2))).astype(np.float32)
vignette /= float(vignette.mean())
stack = np.stack(tiles).astype(np.float32) * brightness * vignette

print(f"Stack shape: {stack.shape}")

In [ ]:
# NumPy backend (always available)
model_np = BaSiC(stack, backend="numpy")
model_np.working_size = TILE
model_np.run()
print(f"NumPy  flat std: {model_np.get_flatfield().std():.4f}")

# Try PyTorch if installed
try:
    import torch as _torch  # noqa: F401

    model_torch = BaSiC(stack, backend="torch")
    model_torch.working_size = TILE
    model_torch.run()
    print(f"Torch  flat std: {model_torch.get_flatfield().std():.4f}")

    diff = np.abs(model_np.get_flatfield() - model_torch.get_flatfield()).max()
    print(f"Max abs diff NumPy vs Torch: {diff:.2e}  (should be < 1e-4)")
except ImportError:
    print("PyTorch not installed — NumPy-only demo. Install with: uv sync --extra gpu")

## 2. Parallel z-fitting

`fit_mosaic()` fits one BaSiC model per z-level.  `n_workers` controls the
thread count.  `None` (default) uses all logical cores minus two.
Set `n_workers=1` to run sequentially.

In [ ]:
import time

GRID, N_Z = (3, 4), 4
y, x = np.mgrid[-1 : 1 : GRID[0] * TILE * 1j, -1 : 1 : GRID[1] * TILE * 1j]  # type: ignore[misc]

reps_y = GRID[0] * TILE // source.shape[0] + 1
reps_x = GRID[1] * TILE // source.shape[1] + 1
tiled = np.tile(source, (reps_y, reps_x))
plane = tiled[: GRID[0] * TILE, : GRID[1] * TILE]
mosaic_arr = np.stack(
    [
        plane
        * rng.uniform(0.4, 0.8, GRID).repeat(TILE, axis=0).repeat(TILE, axis=1)
        * (
            0.5
            + 0.5
            * np.exp(-1.5 * (x[: GRID[0] * TILE, : GRID[1] * TILE] ** 2 + y[: GRID[0] * TILE, : GRID[1] * TILE] ** 2)).astype(
                np.float32
            )
        )
        for _ in range(N_Z)
    ]
).astype(np.float32)

mosaic = MosaicGrid(mosaic_arr, tile_shape=(TILE, TILE))

# Sequential
t0 = time.perf_counter()
fit_seq = fit_mosaic(mosaic, basic_kwargs={"working_size": TILE}, n_workers=1)
t_seq = time.perf_counter() - t0

# Parallel (2 workers)
t0 = time.perf_counter()
fit_par = fit_mosaic(mosaic, basic_kwargs={"working_size": TILE}, n_workers=2)
t_par = time.perf_counter() - t0

print(f"Sequential: {t_seq:.2f}s")
print(f"Parallel 2 workers: {t_par:.2f}s")
diff_ff = np.abs(fit_seq.flatfields - fit_par.flatfields).max()
print(f"Max diff between runs: {diff_ff:.2e}  (should be <1e-5)")

## 3. OME-Zarr I/O

The `linum_basic.io.zarr` module provides `write_ome_zarr()` and
`load_ome_zarr()` for efficient out-of-core storage.  Requires `ome-zarr`
(installed with the `notebooks` extra).

In [ ]:
try:
    import ome_zarr  # noqa: F401

    _has_ome_zarr = True
except ImportError:
    _has_ome_zarr = False
    print("ome-zarr not installed — skipping zarr I/O demo.")
    print("Install with: uv sync --extra notebooks")

In [ ]:
if _has_ome_zarr:
    from linum_basic.io.zarr import load_ome_zarr, write_ome_zarr

    tmpdir = Path(tempfile.mkdtemp())
    zarr_path = tmpdir / "corrected.ome.zarr"

    corrected = apply_fit(mosaic, fit_seq)

    # Write
    write_ome_zarr(
        zarr_path,
        corrected,
        axes=["z", "y", "x"],
        scale=[1.0, 0.5, 0.5],
    )
    print(f"Wrote {corrected.shape} array to {zarr_path}")

    # Read back
    array_rt, axes_rt, scale_rt = load_ome_zarr(zarr_path)
    print(f"Read back: shape={array_rt.shape}  axes={axes_rt}  scale={scale_rt}")
    print(f"Max roundtrip error: {np.abs(corrected - array_rt).max():.2e}")

    shutil.rmtree(tmpdir)

## 4. CLI usage via `subprocess`

Every workflow is also accessible via the `basic` command-line tool.
The four sub-commands are:

```text
basic correct  --input DIR --output DIR [options]
basic fit      --input mosaic.ome.zarr --output corrected.ome.zarr [options]
basic tune     --input mosaic.ome.zarr [options]
basic preview  --input volume.ome.zarr --output preview.png [options]
```

In [ ]:
# Show top-level help
result = subprocess.run(
    ["python", "-m", "linum_basic.cli", "--help"],
    capture_output=True,
    text=True,
)
print(result.stdout or result.stderr)

In [ ]:
# Write a small image stack to a temp dir and run `basic correct`
tmpdir = Path(tempfile.mkdtemp())
input_dir = tmpdir / "input"
output_dir = tmpdir / "output"
input_dir.mkdir()

# Save 20 small tif-like PNG images
import imageio.v3 as iio  # noqa: E402  (lazy import)

img_base = load_sample_image()[:64, :64].astype(np.uint8)
rng2 = np.random.default_rng(3)
for i in range(20):
    bright = rng2.uniform(0.5, 0.9)
    iio.imwrite(input_dir / f"tile_{i:03d}.png", (img_base * bright).astype(np.uint8))

result = subprocess.run(
    [
        "basic",
        "correct",
        "--input",
        str(input_dir),
        "--output",
        str(output_dir),
        "--extension",
        ".png",
    ],
    capture_output=True,
    text=True,
)
print("Return code:", result.returncode)
if result.stdout:
    print(result.stdout[:500])
if result.returncode == 0:
    saved = list(output_dir.glob("*.png"))
    print(f"Output files: {len(saved)}  (expected 20)")

shutil.rmtree(tmpdir)

In [ ]:
# Show `basic correct --help`
result = subprocess.run(
    ["basic", "correct", "--help"],
    capture_output=True,
    text=True,
)
print(result.stdout)

## 5. Visualising results with `linum_basic.viz`

The `viz` module provides publication-ready figures for BaSiC fields,
intensity images, seam metrics, and tuning history.

In [ ]:
from linum_basic import viz

model_viz = BaSiC(stack, backend="numpy")
model_viz.working_size = TILE
model_viz.run()

# show_field: renders a flat-field with auto colorbar
fig, ax = plt.subplots()
viz.show_field(ax, model_viz.get_flatfield(), title="Flat-field")
plt.show()

# show_image: grayscale intensity
fig, ax = plt.subplots()
viz.show_image(ax, stack[0], title="Raw tile 0")
plt.show()

In [ ]:
# field_surface_3d: 3-D surface plot of the flat-field
fig, ax = viz.field_surface_3d(model_viz.get_flatfield(), title="Flat-field surface")
plt.show()

In [ ]:
# aip_preview: average-intensity projection of a volume
volume = mosaic_arr  # (Z, H, W)
fig = viz.aip_preview(volume, title="AIP preview")
plt.show()

## Summary

| Feature | Key API |
|---------|--------|
| Backend selection | `BaSiC(stack, backend="torch", device="cuda:0")` |
| Parallel z-fitting | `fit_mosaic(mosaic, n_workers=8)` |
| OME-Zarr write | `write_ome_zarr(path, array, axes=..., scale=...)` |
| OME-Zarr read | `load_ome_zarr(path)` → `(array, axes, scale)` |
| CLI correct | `basic correct --input DIR --output DIR` |
| CLI fit | `basic fit --input mosaic.ome.zarr --output out.ome.zarr` |
| CLI tune | `basic tune --input mosaic.ome.zarr` |
| CLI preview | `basic preview --input vol.ome.zarr --output preview.png` |